In [1]:
!pip install torch>=2.0.0 torchvision opencv-python insightface onnxruntime scikit-learn numpy pandas tqdm Pillow facenet-pytorch

In [2]:
import cv2
import numpy as np
import random
import re
from pathlib import Path
from itertools import product as iproduct
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
from PIL import Image
from sklearn.metrics import roc_curve, auc as sk_auc

In [3]:
IMAGE_DIR  = "/kaggle/input/datasets/ayushi6/id-images/ID_images_50"   # folder with ID photos
VIDEO_DIR  = "/kaggle/input/datasets/ayushi6/lp-videos"        # folder with videos

# Representative subject to render the detailed card for
# Set to None to auto-pick the highest-quality positive pair
REPRESENTATIVE_SUBJECT = None   # e.g. "GoldieHawn"

SAMPLE_RATE       = 3     # use every Nth frame from each video
QUALITY_THRESHOLD = 0.25  # discard frames below this per video
MIN_FRAMES        = 3     # minimum retained frames before falling back to all
VERIFICATION_THRESHOLD = 0.30   # cosine similarity threshold
RANDOM_SEED       = 42

In [4]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cpu


In [5]:
# STEP 1: LOAD ARCFACE BACKBONE

from insightface.app import FaceAnalysis

print("\n[1/7] Loading pretrained ArcFace model...")
app = FaceAnalysis(
    name="buffalo_l",
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
)
app.prepare(ctx_id=0 if DEVICE == "cuda" else -1, det_size=(640, 640))
print("     ✓ ArcFace (buffalo_l) loaded")


[1/7] Loading pretrained ArcFace model...
download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:03<00:00, 92950.99KB/s]


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)
     ✓ ArcFa

In [6]:
# STEP 2: PARSE FILENAMES

def parse_image_name(filename: str) -> dict | None:
    """
    Pattern: <uid>_<Name>_<age>_<gender>.<ext>
    Example: 1136_HeinrichHertz_21_m.jpg
    """
    stem = Path(filename).stem
    # Strip any double extension like .jpg in .jpg.jpeg
    stem = re.sub(r'\.\w+$', '', stem)
    m = re.match(r'^(\d+)_([A-Za-z]+)_(\d+)_([mf])$', stem)
    if not m:
        return None
    return {
        "uid": m.group(1),
        "name": m.group(2),
        "age": int(m.group(3)),
        "gender": m.group(4),
        "path": filename,
    }
def parse_video_name(filename: str) -> dict | None:
    """
    Pattern: <uid>_<Name>_<age>_<gender>--<activity>.<ext>
    Example: 1135_HeinrichHertz_36_m--speaking.mp4
    """
    stem = Path(filename).stem
    m = re.match(r'^(\d+)_([A-Za-z]+)_(\d+)_([mf])--(.+)$', stem)
    if not m:
        return None
    return {
        "uid": m.group(1),
        "name": m.group(2),
        "age": int(m.group(3)),
        "gender": m.group(4),
        "activity": m.group(5),
        "path": filename,
    }


print("\n[2/7] Scanning directories...")

image_exts = {".jpg", ".jpeg", ".png", ".bmp"}
video_exts = {".mp4", ".avi", ".mov", ".mkv"}

image_dir = Path(IMAGE_DIR)
video_dir = Path(VIDEO_DIR)

images_raw = sorted([f for f in image_dir.iterdir()
                      if f.suffix.lower() in image_exts or
                         ''.join(f.suffixes).lower() in {'.jpg.jpeg', '.jpg.png'}])
videos_raw = sorted([f for f in video_dir.iterdir()
                      if f.suffix.lower() in video_exts])
image_meta = []
for f in images_raw:
    m = parse_image_name(f.name)
    if m:
        m["path"] = str(f)
        image_meta.append(m)
    else:
        print(f"     ⚠ Could not parse image filename: {f.name}")

video_meta = []
for f in videos_raw:
    m = parse_video_name(f.name)
    if m:
        m["path"] = str(f)
        video_meta.append(m)
    else:
        print(f"     ⚠ Could not parse video filename: {f.name}")

print(f"     Found {len(image_meta)} parseable images, {len(video_meta)} parseable videos")



[2/7] Scanning directories...
     Found 50 parseable images, 49 parseable videos


In [7]:
# STEP 3: BUILD PAIRS  (positive + negative)

print("\n[3/7] Building positive and negative pairs...")

# Key: (name, gender) → same person
def person_key(meta):
    return (meta["name"].lower(), meta["gender"])

# Group by person key
from collections import defaultdict
img_by_person = defaultdict(list)
vid_by_person = defaultdict(list)

for m in image_meta:
    img_by_person[person_key(m)].append(m)
for m in video_meta:
    vid_by_person[person_key(m)].append(m)

pairs = []   # list of dicts: {img, vid, label, subject_name}

# --- Positive pairs: matched name+gender ---
matched_keys = set(img_by_person.keys()) & set(vid_by_person.keys())
print(f"     Matched subjects (name+gender): {len(matched_keys)}")

for key in sorted(matched_keys):
    # Take the first image and first video for that person (or all combos)
    img = img_by_person[key][0]
    vid = vid_by_person[key][0]
    pairs.append({
        "img": img, "vid": vid,
        "label": 1,
        "subject": img["name"],
        "pair_type": "positive"
    })

# --- Negative pairs: one per subject — pair with a random *different* person's video ---
all_video_keys = list(vid_by_person.keys())
for key in sorted(matched_keys):
    img = img_by_person[key][0]
    # Pick a different person's video
    neg_keys = [k for k in all_video_keys if k != key]
    if not neg_keys:
        continue
    neg_key = random.choice(neg_keys)
    neg_vid  = random.choice(vid_by_person[neg_key])
    pairs.append({
        "img": img, "vid": neg_vid,
        "label": 0,
        "subject": f"{img['name']} vs {neg_vid['name']}",
        "pair_type": "negative"
    })

print(f"     Total pairs: {len(pairs)}  "
      f"(positive={sum(p['label']==1 for p in pairs)}, "
      f"negative={sum(p['label']==0 for p in pairs)})")



[3/7] Building positive and negative pairs...
     Matched subjects (name+gender): 49
     Total pairs: 98  (positive=49, negative=49)


In [8]:
# STEP 4: PROCESSING HELPERS

def get_embedding_robust(image_bgr: np.ndarray):
    faces = app.get(image_bgr)
    if faces:
        face = max(faces, key=lambda f: f.det_score)
        return face.normed_embedding, float(face.det_score), face
    for scale in [1.5, 2.0, 0.75]:
        h, w = image_bgr.shape[:2]
        resized = cv2.resize(image_bgr, (int(w*scale), int(h*scale)))
        faces = app.get(resized)
        if faces:
            face = max(faces, key=lambda f: f.det_score)
            return face.normed_embedding, float(face.det_score), face

    app.det_model.det_thresh = 0.3
    faces = app.get(image_bgr)
    app.det_model.det_thresh = 0.5
    if faces:
        face = max(faces, key=lambda f: f.det_score)
        return face.normed_embedding, float(face.det_score), face

    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    eq   = cv2.equalizeHist(gray)
    enhanced = cv2.cvtColor(eq, cv2.COLOR_GRAY2BGR)
    faces = app.get(enhanced)
    if faces:
        face = max(faces, key=lambda f: f.det_score)
        return face.normed_embedding, float(face.det_score), face

    h, w  = image_bgr.shape[:2]
    size  = min(h, w)
    y0    = (h - size) // 2
    x0    = (w - size) // 2
    crop  = image_bgr[y0:y0+size, x0:x0+size]
    crop  = cv2.resize(crop, (112, 112))
    emb   = app.models['recognition'].get_feat(crop)
    emb   = emb / (np.linalg.norm(emb) + 1e-8)
    return emb.flatten(), 0.5, None
def laplacian_sharpness(image_bgr):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    return float(1 / (1 + np.exp(-0.01 * (lap_var - 100))))

def face_frontality(landmarks):
    if landmarks is None:
        return 0.5
    left_eye  = landmarks[0]
    right_eye = landmarks[1]
    nose      = landmarks[2]
    eye_center = (left_eye + right_eye) / 2
    eye_width  = np.linalg.norm(right_eye - left_eye)
    nose_offset = abs(nose[0] - eye_center[0]) / (eye_width + 1e-8)
    return float(max(0.0, 1.0 - 2 * nose_offset))

def compute_quality_score(image_bgr, det_score, landmarks):
    sharpness  = laplacian_sharpness(image_bgr)
    frontality = face_frontality(landmarks)
    return float(np.clip(0.4*det_score + 0.4*sharpness + 0.2*frontality, 0, 1))


def process_image(img_path: str):
    bgr = cv2.imread(img_path)
    assert bgr is not None, f"Cannot read {img_path}"
    emb, det_score, face_obj = get_embedding_robust(bgr)
    kps = face_obj.kps if face_obj is not None else None
    age = int(face_obj.age) if (face_obj is not None and hasattr(face_obj, 'age')) else 30
    quality = compute_quality_score(bgr, det_score, kps)
    return {
        "embedding": emb,
        "det_score": det_score,
        "quality": quality,
        "age_est": age,
        "bgr": bgr,
        "face_obj": face_obj,
    }


def process_video(video_path: str):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps          = cap.get(cv2.CAP_PROP_FPS)

    all_frames = []
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % SAMPLE_RATE == 0:
            all_frames.append(frame)
        idx += 1
    cap.release()
    frame_results = []
    for frame_bgr in all_frames:
        emb, det_score, face_obj = get_embedding_robust(frame_bgr)
        kps     = face_obj.kps if face_obj is not None else None
        quality = compute_quality_score(frame_bgr, det_score, kps)
        frame_results.append({
            "det_score": det_score,
            "quality": quality,
            "embedding": emb,
            "frame": frame_bgr,
            "face_obj": face_obj,
        })

    retained  = [r for r in frame_results if r["quality"] >= QUALITY_THRESHOLD]
    discarded = [r for r in frame_results if r["quality"] < QUALITY_THRESHOLD]
    if len(retained) < MIN_FRAMES:
        retained = frame_results
        discarded = []

    qualities  = np.array([r["quality"] for r in retained], dtype=np.float32)
    embeddings = np.stack([r["embedding"] for r in retained], axis=0)
    weights    = qualities / qualities.sum()
    f_video    = (weights[:, None] * embeddings).sum(axis=0)
    f_video    = f_video / (np.linalg.norm(f_video) + 1e-8)
    q_video    = float(qualities.mean())

    mid_obj = retained[len(retained)//2]["face_obj"]
    age_est = int(mid_obj.age) if (mid_obj is not None and hasattr(mid_obj, 'age')) else 30
    return {
        "embedding": f_video,
        "q_video": q_video,
        "age_est": age_est,
        "weights": weights,
        "retained": retained,
        "discarded": discarded,
        "all_frames_count": len(all_frames),
        "total_frames": total_frames,
        "fps": fps,
    }


In [9]:
# STEP 5: RUN ALL PAIRS

print(f"\n[4/7] Processing {len(pairs)} pairs...")
print("      (this may take a few minutes)\n")

results = []

for pi, pair in enumerate(pairs):
    subj = pair["subject"]
    lbl  = pair["label"]
    print(f"  [{pi+1:03d}/{len(pairs)}] {pair['pair_type']:8s} | {subj}")

    # --- Process image ---
    try:
        img_result = process_image(pair["img"]["path"])
    except Exception as e:
        print(f"        ✗ Image error: {e}")
        continue
    # --- Process video ---
    vid_result = process_video(pair["vid"]["path"])
    if vid_result is None:
        print(f"        ✗ Could not open video")
        continue

    # --- Similarity & AQUALR ---
    cos_sim  = float(np.dot(vid_result["embedding"], img_result["embedding"]))
    q_video  = vid_result["q_video"]
    age_diff = abs(img_result["age_est"] - vid_result["age_est"])
    a_norm   = float(np.clip(age_diff / 30.0, 0.0, 1.0))

    difficulty      = (1 - q_video) * 0.6 + a_norm * 0.4
    aqualr          = float(np.clip(np.exp(2.0 * (1 - difficulty)) / np.exp(1.0), 0.1, 10.0))
    adaptive_margin = 0.5 * aqualr

    verified = cos_sim >= VERIFICATION_THRESHOLD
    confidence = float(np.clip(
        50 + 50 * (cos_sim - VERIFICATION_THRESHOLD) / (1 - VERIFICATION_THRESHOLD), 0, 100
    ))

    if q_video >= 0.6 and age_diff <= 5:
        diff_bin = "Easy"
    elif q_video >= 0.35 or age_diff <= 15:
        diff_bin = "Medium"
    else:
        diff_bin = "Hard"
    rec = {
        "subject": subj,
        "pair_type": pair["pair_type"],
        "label": lbl,
        "img_path": pair["img"]["path"],
        "vid_path": pair["vid"]["path"],
        "img_name": pair["img"]["name"],
        "vid_name": pair["vid"]["name"],
        "cos_sim": cos_sim,
        "verified": verified,
        "correct": (verified == bool(lbl)),
        "confidence": confidence,
        "q_video": q_video,
        "img_quality": img_result["quality"],
        "age_diff": age_diff,
        "aqualr": aqualr,
        "adaptive_margin": adaptive_margin,
        "diff_bin": diff_bin,
        # Store for representative card
        "_img_result": img_result,
        "_vid_result": vid_result,
    }
    results.append(rec)
    print(f"        sim={cos_sim:.4f}  {'✓ MATCH' if verified else '✗ NO MATCH'}  "
          f"label={'same' if lbl else 'diff'}  correct={'✓' if rec['correct'] else '✗'}")

print(f"\n  Done — {len(results)} pairs processed successfully")


[4/7] Processing 98 pairs...
      (this may take a few minutes)

  [001/98] positive | AdolfHitlerr
        sim=0.5923  ✓ MATCH  label=same  correct=✓
  [002/98] positive | akistsoxatzopoulos
        sim=0.0823  ✗ NO MATCH  label=same  correct=✗
  [003/98] positive | AlKapone
        sim=0.2755  ✗ NO MATCH  label=same  correct=✗
  [004/98] positive | AndreasPapantreou
        sim=-0.0727  ✗ NO MATCH  label=same  correct=✗
  [005/98] positive | angelamerkel
        sim=0.3736  ✓ MATCH  label=same  correct=✓
  [006/98] positive | BerryGordy
        sim=0.5927  ✓ MATCH  label=same  correct=✓
  [007/98] positive | ConradHilton
        sim=0.9696  ✓ MATCH  label=same  correct=✓
  [008/98] positive | DavidSarnoff
        sim=0.2391  ✗ NO MATCH  label=same  correct=✗
  [009/98] positive | ElVenizelos
        sim=0.3853  ✓ MATCH  label=same  correct=✓
  [010/98] positive | ElvisPresley
        sim=0.4398  ✓ MATCH  label=same  correct=✓
  [011/98] positive | EsteeLauder
        sim=0.5318  ✓ 

In [10]:
# STEP 6: AGGREGATE METRICS

print("\n[5/7] Computing dataset-level metrics...")

labels_all   = np.array([r["label"]   for r in results])
scores_all   = np.array([r["cos_sim"] for r in results])
verified_all = np.array([r["verified"] for r in results])
correct_all  = np.array([r["correct"]  for r in results])

n_pos = int(labels_all.sum())
n_neg = int((1 - labels_all).sum())
n_total = len(results)

accuracy = float(correct_all.mean()) * 100

# TP / FP / TN / FN
TP = int(((verified_all == 1) & (labels_all == 1)).sum())
FP = int(((verified_all == 1) & (labels_all == 0)).sum())
TN = int(((verified_all == 0) & (labels_all == 0)).sum())
FN = int(((verified_all == 0) & (labels_all == 1)).sum())
TAR = TP / (TP + FN) if (TP + FN) > 0 else 0.0  # true accept rate
FAR = FP / (FP + TN) if (FP + TN) > 0 else 0.0  # false accept rate
TRR = TN / (TN + FP) if (TN + FP) > 0 else 0.0  # true reject rate
FRR = FN / (FN + TP) if (FN + TP) > 0 else 0.0  # false reject rate

# ROC / AUC
fpr, tpr, thresholds = roc_curve(labels_all, scores_all)
roc_auc = sk_auc(fpr, tpr)

# TAR @ FAR=1%, FAR=10%
def tar_at_far(fpr_arr, tpr_arr, target_far):
    idx = np.searchsorted(fpr_arr, target_far)
    if idx >= len(tpr_arr):
        return float(tpr_arr[-1])
    return float(tpr_arr[idx])

tar1  = tar_at_far(fpr, tpr, 0.01)
tar10 = tar_at_far(fpr, tpr, 0.10)

# Quality / difficulty breakdown
by_diff = {"Easy": [], "Medium": [], "Hard": []}
for r in results:
    by_diff[r["diff_bin"]].append(r)
print(f"\n  {'='*52}")
print(f"  DATASET-LEVEL METRICS")
print(f"  {'='*52}")
print(f"  Total pairs:         {n_total}  (pos={n_pos}, neg={n_neg})")
print(f"  Accuracy:            {accuracy:.1f}%")
print(f"  TAR (sensitivity):   {TAR*100:.1f}%")
print(f"  FAR (1-specificity): {FAR*100:.1f}%")
print(f"  TRR (specificity):   {TRR*100:.1f}%")
print(f"  FRR (miss rate):     {FRR*100:.1f}%")
print(f"  AUC (ROC):           {roc_auc:.4f}")
print(f"  TAR @ FAR=1%:        {tar1*100:.1f}%")
print(f"  TAR @ FAR=10%:       {tar10*100:.1f}%")
print(f"  TP={TP}  FP={FP}  TN={TN}  FN={FN}")
print(f"  {'='*52}")



[5/7] Computing dataset-level metrics...

  DATASET-LEVEL METRICS
  Total pairs:         98  (pos=49, neg=49)
  Accuracy:            83.7%
  TAR (sensitivity):   67.3%
  FAR (1-specificity): 0.0%
  TRR (specificity):   100.0%
  FRR (miss rate):     32.7%
  AUC (ROC):           0.9738
  TAR @ FAR=1%:        89.8%
  TAR @ FAR=10%:       93.9%
  TP=33  FP=0  TN=49  FN=16


In [11]:
# STEP 7: PICK REPRESENTATIVE SUBJECT

positives = [r for r in results if r["label"] == 1]

if REPRESENTATIVE_SUBJECT is not None:
    rep_candidates = [r for r in positives
                      if r["img_name"].lower() == REPRESENTATIVE_SUBJECT.lower()]
    rep = rep_candidates[0] if rep_candidates else None
else:
    rep = None

if rep is None:
    # Auto: highest-quality correctly-classified positive pair
    correct_pos = [r for r in positives if r["correct"]]
    if correct_pos:
        rep = max(correct_pos, key=lambda r: r["q_video"])
    else:
        rep = max(positives, key=lambda r: r["q_video"]) if positives else results[0]

print(f"\n[6/7] Representative subject: {rep['subject']}  (q_video={rep['q_video']:.3f})")


[6/7] Representative subject: MargaretThatcher  (q_video=0.706)


In [12]:
import matplotlib.gridspec as gridspec

print("\n[7/7] Generating final metrics and subject visualization...")

fig_rep = plt.figure(figsize=(10, 12))
fig_rep.patch.set_facecolor(BG)

gs_rep = gridspec.GridSpec(
    3, 2,
    figure=fig_rep,
    height_ratios=[0.4, 1, 1],
    hspace=0.3,
    wspace=0.2
)

# ── 1. Metrics Card ───────────────────────────────────────
ax_metrics = fig_rep.add_subplot(gs_rep[0, :])
ax_metrics.set_facecolor(CARD_BG)

for sp in ax_metrics.spines.values():
    sp.set_visible(True)
    sp.set_edgecolor(BLUE)
    sp.set_linewidth(1.5)

ax_metrics.set_xticks([])
ax_metrics.set_yticks([])

result_txt  = "✓ MATCH" if rep["verified"] else "✗ NO MATCH"
correct_txt = "✓ Correct" if rep["correct"] else "✗ Wrong"

metrics_str = (
    f"--- DATASET GLOBAL METRICS ---\n"
    f"Accuracy: {accuracy:.1f}%   |   AUC: {roc_auc:.4f}   |   TAR @ FAR=1%: {tar1*100:.1f}%\n\n"
    f"--- REPRESENTATIVE SUBJECT ---\n"
    f"Subject: {rep['subject']}\n"
    f"Sim: {rep['cos_sim']:.4f}   |   {result_txt}   |   {correct_txt}\n"
    f"Difficulty: {rep['diff_bin']}   |   AQUALR: {rep['aqualr']:.3f}   |   Confidence: {rep['confidence']:.1f}%"
)

ax_metrics.text(
    0.5, 0.5, metrics_str,
    ha="center", va="center",
    color=FG, fontsize=12, fontweight="bold",
    linespacing=1.5
)

# ── 2. Extract Frames ─────────────────────────────────────
rep_img = rep["_img_result"]
rep_vid = rep["_vid_result"]

rep_retained = rep_vid.get("retained", [])

if len(rep_retained) == 0:
    print("⚠️ No retained frames found")
    best_frame = med_frame = worst_frame = rep_img["bgr"]
    best_q = med_q = worst_q = 0.0
else:
    sorted_ret = sorted(rep_retained, key=lambda r: r["quality"], reverse=True)

    best_frame  = sorted_ret[0]["frame"]
    med_frame   = sorted_ret[len(sorted_ret)//2]["frame"]
    worst_frame = sorted_ret[-1]["frame"]

    best_q  = sorted_ret[0]["quality"]
    med_q   = sorted_ret[len(sorted_ret)//2]["quality"]
    worst_q = sorted_ret[-1]["quality"]

# ── 3. Plot Images ───────────────────────────────────────

# ID Image
ax_id = fig_rep.add_subplot(gs_rep[1, 0])
ax_id.imshow(bgr_rgb(crop_face(rep_img["bgr"])))
ax_id.set_title(f"ID: {rep['img_name']}\nq={rep_img['quality']:.3f}", color=FG, fontsize=11)
ax_id.axis("off")
for sp in ax_id.spines.values():
    sp.set_visible(True); sp.set_edgecolor(BLUE); sp.set_linewidth(2)

# Best Frame
ax_best = fig_rep.add_subplot(gs_rep[1, 1])
ax_best.imshow(bgr_rgb(crop_face(best_frame)))
ax_best.set_title(f"Best frame\nq={best_q:.3f}", color=FG, fontsize=11)
ax_best.axis("off")
for sp in ax_best.spines.values():
    sp.set_visible(True); sp.set_edgecolor(GREEN); sp.set_linewidth(2)

# Median Frame
ax_med = fig_rep.add_subplot(gs_rep[2, 0])
ax_med.imshow(bgr_rgb(crop_face(med_frame)))
ax_med.set_title(f"Median frame\nq={med_q:.3f}", color=FG, fontsize=11)
ax_med.axis("off")
for sp in ax_med.spines.values():
    sp.set_visible(True); sp.set_edgecolor(YELLOW); sp.set_linewidth(2)

# Worst Frame
ax_worst = fig_rep.add_subplot(gs_rep[2, 1])
ax_worst.imshow(bgr_rgb(crop_face(worst_frame)))
ax_worst.set_title(f"Worst retained\nq={worst_q:.3f}", color=FG, fontsize=11)
ax_worst.axis("off")
for sp in ax_worst.spines.values():
    sp.set_visible(True); sp.set_edgecolor(RED); sp.set_linewidth(2)

plt.tight_layout(pad=3.0)
plt.savefig("global_metrics_and_subject.png", dpi=150, facecolor=BG)
plt.show()

print("\nFigure saved → global_metrics_and_subject.png")


[7/7] Generating final metrics and subject visualization...


NameError: name 'BG' is not defined

<Figure size 1000x1200 with 0 Axes>

In [ ]:
# SUMMARY 

print("\n" + "="*62)
print("  PAPER-READY BATCH RESULT SUMMARY — AQUAFace-V")
print("="*62)
print(f"  Backbone:            ResNet50 (ArcFace, buffalo_l), 512-d")
print(f"  Aggregation:         Quality-Weighted Mean (SER-FIQ proxy)")
print(f"  Threshold:           {VERIFICATION_THRESHOLD}  (@FAR≈1% tuned on dataset)")
print(f"  Sample rate:         every {SAMPLE_RATE}rd frame")
print(f"  ---")
print(f"  Total pairs:         {n_total}  (pos={n_pos}, neg={n_neg})")
print(f"  ---")
print(f"  Accuracy:            {accuracy:.2f}%")
print(f"  TAR  (sensitivity):  {TAR*100:.2f}%")
print(f"  FAR  (1-specificity):{FAR*100:.2f}%")
print(f"  TRR  (specificity):  {TRR*100:.2f}%")
print(f"  FRR  (miss rate):    {FRR*100:.2f}%")
print(f"  AUC  (ROC):          {roc_auc:.4f}")
print(f"  TAR @ FAR=1%:        {tar1*100:.2f}%")
print(f"  TAR @ FAR=10%:       {tar10*100:.2f}%")
print(f"  ---")
print(f"  Difficulty breakdown:")
for d in ["Easy", "Medium", "Hard"]:
    grp = by_diff[d]
    if grp:
        acc = np.mean([r["correct"] for r in grp]) * 100
        avg_q = np.mean([r["q_video"] for r in grp])
        print(f"    {d:8s}: n={len(grp):3d}  acc={acc:.1f}%  avg_q_video={avg_q:.3f}")
print(f"  ---")
print(f"  Representative: {rep['subject']}")
print(f"    sim={rep['cos_sim']:.4f}  AQUALR={rep['aqualr']:.3f}  "
      f"Difficulty={rep['diff_bin']}  {'✓ Correct' if rep['correct'] else '✗ Wrong'}")
print("="*62)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from itertools import product as iproduct
from matplotlib.patches import Patch

# ── Universal Styling & Colors ──────────────────────────────
BG       = "#0f0f1a"
CARD_BG  = "#1a1a2e"
FG       = "#e8e8f0"
GREEN    = "#00ff88"
RED      = "#ff4466"
BLUE     = "#4488ff"
YELLOW   = "#ffcc00"
PURPLE   = "#bb88ff"

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor":   CARD_BG,
    "text.color":       FG,
    "axes.labelcolor":  FG,
    "xtick.color":      FG,
    "ytick.color":      FG,
    "axes.edgecolor":   "#333355",
})

def style_ax(ax, border_color):
    ax.set_facecolor(CARD_BG)
    for sp in ax.spines.values():
        sp.set_visible(True)
        sp.set_edgecolor(border_color)
        sp.set_linewidth(1.5)
    ax.tick_params(colors=FG, labelsize=9)

In [ ]:
fig_roc, ax_roc = plt.subplots(figsize=(8, 6))
style_ax(ax_roc, BLUE)

ax_roc.plot(fpr, tpr, color=GREEN, linewidth=2.5, label=f"AUC = {roc_auc:.4f}")
ax_roc.plot([0, 1], [0, 1], color="#555577", linewidth=1, linestyle="--", label="Random")
ax_roc.axvline(0.01, color=YELLOW, linewidth=1, linestyle=":", alpha=0.7, label="FAR=1%")
ax_roc.axvline(0.10, color=PURPLE, linewidth=1, linestyle=":", alpha=0.7, label="FAR=10%")

ax_roc.set_xlabel("False Accept Rate (FAR)", fontsize=10)
ax_roc.set_ylabel("True Accept Rate (TAR)", fontsize=10)
ax_roc.set_title("ROC Curve", color=FG, fontsize=12)
ax_roc.legend(fontsize=9, facecolor=CARD_BG, labelcolor=FG)
ax_roc.set_xlim()
ax_roc.set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig("chart_1_roc_curve.png", dpi=150, facecolor=BG)
plt.show()

In [ ]:
fig_dist, ax_dist = plt.subplots(figsize=(8, 6))
style_ax(ax_dist, PURPLE)

pos_scores = scores_all[labels_all == 1]
neg_scores = scores_all[labels_all == 0]
bins = np.linspace(scores_all.min() - 0.05, scores_all.max() + 0.05, 30)

ax_dist.hist(pos_scores, bins=bins, color=GREEN, alpha=0.75, label=f"Same person (n={len(pos_scores)})")
ax_dist.hist(neg_scores, bins=bins, color=RED,   alpha=0.75, label=f"Different (n={len(neg_scores)})")
ax_dist.axvline(VERIFICATION_THRESHOLD, color=YELLOW, linewidth=2, linestyle="--", label=f"Threshold ({VERIFICATION_THRESHOLD})")

ax_dist.set_xlabel("Cosine Similarity", fontsize=10)
ax_dist.set_ylabel("Count", fontsize=10)
ax_dist.set_title("Score Distribution", color=FG, fontsize=12)
ax_dist.legend(fontsize=9, facecolor=CARD_BG, labelcolor=FG)

plt.tight_layout()
plt.savefig("chart_2_score_distribution.png", dpi=150, facecolor=BG)
plt.show()

In [ ]:
fig_cm, ax_cm = plt.subplots(figsize=(6, 6))
style_ax(ax_cm, BLUE)

cm_data = np.array([[TN, FP], [FN, TP]])
im = ax_cm.imshow(cm_data, cmap="Blues", vmin=0)

ax_cm.set_xticks([0, 1])
ax_cm.set_yticks([0, 1])
ax_cm.set_xticklabels(["Predicted\nDiff", "Predicted\nSame"], fontsize=10, color=FG)
ax_cm.set_yticklabels(["Actual\nDiff", "Actual\nSame"], fontsize=10, color=FG)
ax_cm.set_title("Confusion Matrix", color=FG, fontsize=12)

for i, j in iproduct(range(2), range(2)):
    ax_cm.text(j, i, str(cm_data[i, j]),
               ha="center", va="center",
               color=BG if cm_data[i, j] > cm_data.max()/2 else FG,
               fontsize=22, fontweight="bold")

plt.tight_layout()
plt.savefig("chart_3_confusion_matrix.png", dpi=150, facecolor=BG)
plt.show()

In [ ]:
fig_diff, ax_diff = plt.subplots(figsize=(8, 6))
style_ax(ax_diff, YELLOW)

diff_labels  = ["Easy", "Medium", "Hard"]
diff_colors  = [GREEN, YELLOW, RED]
diff_accs    = []
diff_counts  = []

for d in diff_labels:
    grp = by_diff[d]
    if grp:
        acc = np.mean([r["correct"] for r in grp]) * 100
    else:
        acc = 0.0
    diff_accs.append(acc)
    diff_counts.append(len(grp))

bars = ax_diff.bar(diff_labels, diff_accs, color=diff_colors, alpha=0.85, width=0.5)

ax_diff.set_ylim(0, 115)
ax_diff.set_ylabel("Accuracy (%)", fontsize=10)
ax_diff.set_title("Accuracy by Difficulty Bin", color=FG, fontsize=12)

for bar, acc, cnt in zip(bars, diff_accs, diff_counts):
    ax_diff.text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 2,
                 f"{acc:.0f}%\n(n={cnt})",
                 ha="center", color=FG, fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig("chart_4_accuracy_by_difficulty.png", dpi=150, facecolor=BG)
plt.show()

In [ ]:
fig_strip, ax_strip = plt.subplots(figsize=(14, 5))
style_ax(ax_strip, PURPLE)

pos_results = [r for r in results if r["label"] == 1]
neg_results = [r for r in results if r["label"] == 0]

for i, r in enumerate(pos_results):
    color = GREEN if r["correct"] else RED
    ax_strip.bar(i, r["cos_sim"], color=color, alpha=0.8, width=0.8)

for i, r in enumerate(neg_results):
    color = GREEN if r["correct"] else RED
    ax_strip.bar(i + len(pos_results) + 2, r["cos_sim"], color=color, alpha=0.8, width=0.8)

ax_strip.axhline(VERIFICATION_THRESHOLD, color=YELLOW, linewidth=1.5,
                 linestyle="--", label=f"Threshold ({VERIFICATION_THRESHOLD})")
ax_strip.axvline(len(pos_results) + 0.5, color="#555577", linewidth=1.5, linestyle=":")

# Adaptive text placement based on y-limits
y_text = ax_strip.get_ylim() if ax_strip.get_ylim() > 0.1 else 0.8
ax_strip.text(len(pos_results)/2, y_text, "Positive pairs", ha="center", color=GREEN, fontsize=10, alpha=0.7)
ax_strip.text(len(pos_results) + 2 + len(neg_results)/2, y_text, "Negative pairs", ha="center", color=RED, fontsize=10, alpha=0.7)

ax_strip.set_xlabel("Pair index", fontsize=10)
ax_strip.set_ylabel("Cosine Similarity", fontsize=10)
ax_strip.set_title("Per-Pair Similarity (Green = Correct, Red = Wrong)", color=FG, fontsize=12)
legend_els = [Patch(facecolor=GREEN, label="Correct"), Patch(facecolor=RED, label="Wrong"),
              plt.Line2D([0], [0], color=YELLOW, linestyle="--", label=f"Threshold")]

ax_strip.legend(handles=legend_els, fontsize=9, facecolor=CARD_BG, labelcolor=FG, loc='lower right')

plt.tight_layout()
plt.savefig("chart_5_per_pair_strip.png", dpi=150, facecolor=BG)
plt.show()